---
## 1. Setup & Dependencies

In [ ]:
# Install required packages
!pip install -q kaggle gensim xgboost lightgbm wordcloud plotly
!pip install -q contractions

In [ ]:
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import contractions

# sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve, accuracy_score
)
from sklearn.preprocessing import LabelEncoder

# XGBoost / LightGBM
from xgboost import XGBClassifier
import lightgbm as lgb

# Gensim Word2Vec
from gensim.models import Word2Vec

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, LSTM, Bidirectional, Embedding,
    GlobalAveragePooling1D, BatchNormalization, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TF version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU'))} device(s)")

# Download NLTK resources
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'omw-1.4', 'punkt_tab']:
    nltk.download(resource, quiet=True)

---
## 2. Load the Dataset

In [ ]:
from google.colab import files
print("Upload your kaggle.json API key:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews --unzip

df = pd.read_csv('IMDB Dataset.csv')

print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df['sentiment'].value_counts())
df.head(3)

---
## 3. Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))

# Keep negation words — they carry strong sentiment signal
KEEP_WORDS = {
    'not', 'no', 'nor', 'neither', 'never', 'nothing', 'nowhere',
    'nobody', 'none', "n't", 'cannot', "isn't", "wasn't", "aren't",
    "weren't", "don't", "doesn't", "didn't", "won't", "wouldn't",
    "shouldn't", "couldn't", "hadn't", "haven't", "hasn't"
}
STOP_WORDS -= KEEP_WORDS


def clean_text(text: str, lemmatize: bool = True) -> str:
    """Full preprocessing pipeline for a single review."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 3. Expand contractions  (don't → do not)
    text = contractions.fix(text)
    # 4. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # 5. Remove special characters / punctuation (keep apostrophes for n't)
    text = re.sub(r"[^a-z\s']", ' ', text)
    # 6. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # 7. Tokenise
    tokens = word_tokenize(text)
    # 8. Remove stopwords (but keep negations)
    tokens = [t for t in tokens if t not in STOP_WORDS or t in KEEP_WORDS]
    # 9. Lemmatise
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    # 10. Drop very short tokens
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)


print("Preprocessing all reviews (this may take 2–3 min)...")
t0 = time.time()
df['clean_review'] = df['review'].apply(clean_text)
print(f"Done in {time.time()-t0:.1f}s")

# Encode labels
df['label'] = (df['sentiment'] == 'positive').astype(int)

print("\nBefore vs After:")
idx = 0
print("ORIGINAL:", df['review'].iloc[idx][:200])
print("\nCLEAN   :", df['clean_review'].iloc[idx][:200])

In [ ]:
# Quick sanity check
print(f"Null clean reviews: {df['clean_review'].isna().sum()}")
print(f"Empty clean reviews: {(df['clean_review'].str.strip() == '').sum()}")
df = df[df['clean_review'].str.strip() != ''].reset_index(drop=True)
print(f"Final dataset size: {len(df)}")

---
## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class balance
df['sentiment'].value_counts().plot(
    kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'], edgecolor='black'
)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():,}', (p.get_x()+0.3, p.get_height()+100))

# Review length distribution (tokens after cleaning)
df['token_count'] = df['clean_review'].apply(lambda x: len(x.split()))
for sentiment, color, label in [('positive','#2ecc71','Positive'), ('negative','#e74c3c','Negative')]:
    subset = df[df['sentiment'] == sentiment]['token_count']
    axes[1].hist(subset, bins=50, alpha=0.6, color=color, label=label)
axes[1].set_title('Token Count Distribution (after cleaning)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Tokens')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].set_xlim(0, 600)

# Top words
all_words = ' '.join(df['clean_review']).split()
word_freq = Counter(all_words).most_common(20)
words, freqs = zip(*word_freq)
axes[2].barh(words[::-1], freqs[::-1], color='steelblue')
axes[2].set_title('Top 20 Words (after cleaning)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

print(f"\nToken count stats:")
print(df['token_count'].describe().round(1))

In [ ]:
# Word clouds for each class
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, sentiment, title, cmap in zip(
    axes,
    ['positive', 'negative'],
    ['Positive Reviews', 'Negative Reviews'],
    ['Greens', 'Reds']
):
    text = ' '.join(df[df['sentiment'] == sentiment]['clean_review'])
    wc = WordCloud(
        width=800, height=400, max_words=150,
        background_color='white', colormap=cmap
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 5. Train / Validation / Test Split

In [ ]:
X = df['clean_review'].values
y = df['label'].values

# 70 / 15 / 15 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Train : {len(X_train):,} samples")
print(f"Val   : {len(X_val):,} samples")
print(f"Test  : {len(X_test):,} samples")

# Storage for final comparison
results = {}

---
## 6. TF-IDF + Classical Machine Learning

We vectorise with **TF-IDF** (unigrams + bigrams) and train several models.

In [ ]:
# ── TF-IDF Vectoriser ─────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),       # unigrams + bigrams
    max_features=150_000,
    sublinear_tf=True,        # apply log-normalisation
    min_df=2,
    max_df=0.95,
    analyzer='word'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, fit=True):
    """Fit (optionally) and evaluate a model. Returns F1 macro."""
    if fit:
        model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    f1  = f1_score(y_te, y_pred, average='macro')
    acc = accuracy_score(y_te, y_pred)
    print(f"\n{'='*55}")
    print(f" {name}")
    print(f"{'='*55}")
    print(classification_report(y_te, y_pred, target_names=['Negative','Positive']))
    return f1, acc, model


# ── Logistic Regression ────────────────────────────────────────────────────
lr = LogisticRegression(
    C=5.0, max_iter=1000, solver='lbfgs',
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
f1, acc, lr = evaluate_model(
    'Logistic Regression (TF-IDF)',
    lr, X_train_tfidf, y_train, X_test_tfidf, y_test
)
results['LR + TF-IDF'] = {'F1': f1, 'Acc': acc}

In [ ]:
# ── Linear SVM ────────────────────────────────────────────────────────────
svm = LinearSVC(
    C=1.0, max_iter=5000, class_weight='balanced', random_state=SEED
)
f1, acc, svm = evaluate_model(
    'Linear SVM (TF-IDF)',
    svm, X_train_tfidf, y_train, X_test_tfidf, y_test
)
results['SVM + TF-IDF'] = {'F1': f1, 'Acc': acc}

In [ ]:
# ── Complement Naive Bayes (strong baseline for text) ─────────────────────
from sklearn.preprocessing import MaxAbsScaler
from scipy.sparse import csr_matrix

cnb = ComplementNB(alpha=0.1)
# CNB needs non-negative input — TF-IDF with sublinear_tf is fine
f1, acc, cnb = evaluate_model(
    'Complement Naive Bayes (TF-IDF)',
    cnb, X_train_tfidf, y_train, X_test_tfidf, y_test
)
results['CNB + TF-IDF'] = {'F1': f1, 'Acc': acc}

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────
lgbm = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
f1, acc, lgbm = evaluate_model(
    'LightGBM (TF-IDF)',
    lgbm, X_train_tfidf, y_train, X_test_tfidf, y_test
)
results['LightGBM + TF-IDF'] = {'F1': f1, 'Acc': acc}

In [ ]:
# ── Voting Ensemble (LR + SVM + LightGBM) ─────────────────────────────────
# For soft voting we need probability estimates — SVM needs CalibratedClassifierCV
from sklearn.calibration import CalibratedClassifierCV

svm_cal = CalibratedClassifierCV(
    LinearSVC(C=1.0, max_iter=5000, class_weight='balanced', random_state=SEED),
    cv=3
)

ensemble = VotingClassifier(
    estimators=[
        ('lr',   LogisticRegression(C=5.0, max_iter=1000, solver='lbfgs',
                                    class_weight='balanced', random_state=SEED, n_jobs=-1)),
        ('svm',  svm_cal),
        ('lgbm', lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                     num_leaves=63, class_weight='balanced',
                                     random_state=SEED, n_jobs=-1, verbose=-1))
    ],
    voting='soft',
    weights=[2, 1, 1]
)

f1, acc, ensemble = evaluate_model(
    'Soft Voting Ensemble (TF-IDF)',
    ensemble, X_train_tfidf, y_train, X_test_tfidf, y_test
)
results['Ensemble + TF-IDF'] = {'F1': f1, 'Acc': acc}

In [ ]:
# Confusion Matrix for best classical model
best_classical = ensemble   # replace with whichever is best after running above
y_pred_best = best_classical.predict(X_test_tfidf)

cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Best Classical Model')
plt.tight_layout()
plt.show()

# Word2vec

In [ ]:
# Tokenise for Word2Vec
all_tokens = [review.split() for review in df['clean_review']]

EMBEDDING_DIM = 200

print("Training Word2Vec (Skip-gram)...")
t0 = time.time()
w2v_model = Word2Vec(
    sentences=all_tokens,
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=3,
    sg=1,              # Skip-gram
    workers=4,
    epochs=10,
    seed=SEED
)
print(f"Done in {time.time()-t0:.1f}s  |  Vocab size: {len(w2v_model.wv):,}")

# Sanity check
print("\nWords similar to 'great':")
for word, score in w2v_model.wv.most_similar('great', topn=5):
    print(f"  {word:<20} {score:.4f}")

In [ ]:
# ── Build averaged document vectors ───────────────────────────────────────
def doc_to_vec(tokens, model, dim=EMBEDDING_DIM):
    """Mean-pool word vectors for a document."""
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)


train_tokens = [r.split() for r in X_train]
val_tokens   = [r.split() for r in X_val]
test_tokens  = [r.split() for r in X_test]

X_train_w2v = np.array([doc_to_vec(t, w2v_model) for t in train_tokens])
X_val_w2v   = np.array([doc_to_vec(t, w2v_model) for t in val_tokens])
X_test_w2v  = np.array([doc_to_vec(t, w2v_model) for t in test_tokens])

print(f"Document vector shapes → Train: {X_train_w2v.shape}, Test: {X_test_w2v.shape}")